# 🏆 نظام تصنيف لاعبي الفانتازي باستخدام الشبكات العصبية (ANN)
## المرحلة 1: تحميل البيانات وتجهيزها (Data Loading & Preprocessing)
---
في هذه المرحلة سنقوم بتحميل بيانات الدوري الإنجليزي 2025/26 وتنظيفها وإنشاء فئات التصنيف المطلوبة.

In [ ]:
# 1. تثبيت واستيراد المكتبات
!pip install kagglehub tensorflow scikit-learn pandas numpy matplotlib seaborn

import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob

In [ ]:
# 2. تحميل البيانات من Kaggle
path = kagglehub.dataset_download('calvinrostanto/fantasy-premier-league-2025-2026')
print('✅ Path to dataset files:', path)

# البحث عن ملفات الـ CSV
csv_files = glob.glob(os.path.join(path, '**', 'players.csv'), recursive=True)
if not csv_files:
    csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)

df_raw = pd.read_csv(csv_files[0])
print(f'📊 تم تحميل البيانات بنجاح. الأبعاد: {df_raw.shape}')
df_raw.head()

## 🧹 تنظيف البيانات (Cleaning)
سنقوم بإزالة القيم الفارغة، التكرارات، والأعمدة غير الضرورية.

In [ ]:
df = df_raw.copy()

# 1. إزالة التكرارات
df = df.drop_duplicates()

# 2. اختيار الأعمدة الهامة للتحليل
# ملاحظة: سنحتفظ بالأعمدة الرقمية الأساسية
cols_to_keep = [
    'total_points', 'minutes', 'goals_scored', 'assists', 
    'clean_sheets', 'goals_conceded', 'saves', 'bonus', 'bps', 
    'influence', 'creativity', 'threat', 'ict_index', 'now_cost', 
    'form', 'selected_by_percent', 'element_type'
]
df = df[cols_to_keep]

# 3. معالجة القيم المفقودة (Filling missing values)
df = df.fillna(0)

print(f'✅ البيانات بعد التنظيف: {df.shape}')

## 🛠️ هندسة الميزات وإنشاء الأهداف (Target Engineering)
سنقوم الآن بإنشاء الثلاث تصنيفات المطلوبة:

In [ ]:
# 1. هدف فئة النقاط (Points Tier)
# Star: +8, Standard: 2-7, Blank: 0-1
def classify_points(pts):
    if pts >= 8: return 'Star'
    elif pts >= 2: return 'Standard'
    else: return 'Blank'

df['points_tier'] = df['total_points'].apply(classify_points)

# 2. هدف قيمة السعر (Value for Money)
# سنستخدم معدل النقاط لكل مليون
df['value_ratio'] = df['total_points'] / (df['now_cost'] + 1) # +1 لتجنب القسمة على صفر
q1, q2 = df['value_ratio'].quantile([0.33, 0.66])

def classify_value(val):
    if val <= q1: return 'Overpriced'
    elif val <= q2: return 'Fairly priced'
    else: return 'Underpriced'

df['value_tier'] = df['value_ratio'].apply(classify_value)

# 3. هدف الكلين شيت (Clean Sheet Potential - للمدافعين والحراس فقط)
# سنعتبر High CS لو اللاعب جاب أكتر من 5 كلين شيت في الموسم
def classify_cs(cs):
    return 'High CS Chance' if cs >= 5 else 'Low CS Chance'

df['cs_potential'] = df['clean_sheets'].apply(classify_cs)

print('✅ تم إنشاء الأعمدة المستهدفة (Targets) بنجاح.')

In [ ]:
# عرض توزيع الفئات للتأكد من التوازن
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df, x='points_tier', ax=axes[0], palette='viridis')
axes[0].set_title('توزيع فئات النقاط')

sns.countplot(data=df, x='value_tier', ax=axes[1], palette='magma')
axes[1].set_title('توزيع فئات القيمة')

sns.countplot(data=df, x='cs_potential', ax=axes[2], palette='coolwarm')
axes[2].set_title('توزيع احتمالية الكلين شيت')

plt.tight_layout()
plt.show()